# Supported-Cavity Comparison

Compare the archived deep-cavity sweeps against the updated `100 m` ocean cavity geometry with a solid support layer beneath the water column. The matched cases reuse the same source frequencies (`10`, `5`, and `1 Hz`) and a smaller shared set of material parameter combinations so we can isolate the effect of adding material under the cavity.

In [ ]:
from csv import DictReader
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

ROOT = Path.cwd()
old_paths = {
    10: ROOT / "results_10hz" / "summary.csv",
    5: ROOT / "results_5hz" / "summary.csv",
    1: ROOT / "results_1hz" / "summary.csv",
}
new_paths = {
    10: ROOT / "results_supportedcavity_10hz" / "summary.csv",
    5: ROOT / "results_supportedcavity_5hz" / "summary.csv",
    1: ROOT / "results_supportedcavity_1hz" / "summary.csv",
}

def load_summary(path):
    with path.open() as handle:
        return {row["case_id"]: row for row in DictReader(handle)}

old = {freq: load_summary(path) for freq, path in old_paths.items()}
new = {freq: load_summary(path) for freq, path in new_paths.items()}
common_case_ids = sorted(set.intersection(*[set(old[freq]) & set(new[freq]) for freq in old]))
common_case_ids

In [ ]:
def vec(dataset, freq, key, cast=float):
    return np.asarray([cast(dataset[freq][case_id][key]) for case_id in common_case_ids])

def labels_for_case(case_id, freq=10):
    row = new[freq][case_id]
    return f"{case_id}\nwf={float(row['water_fraction']):.2f}, ss={float(row['shear_scale']):.2f}"

for case_id in common_case_ids:
    old_r = [float(old[freq][case_id]["reflection_coefficient"]) for freq in (10, 5, 1)]
    new_r = [float(new[freq][case_id]["reflection_coefficient"]) for freq in (10, 5, 1)]
    print(f"{case_id:>12s}  old={old_r}  new={new_r}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for ax, freq in zip(axes, (10, 5, 1)):
    x = vec(old, freq, "reflection_coefficient")
    y = vec(new, freq, "reflection_coefficient")
    lo = min(x.min(), y.min())
    hi = max(x.max(), y.max())
    pad = 0.05 * max(1e-6, hi - lo)
    ax.plot([lo - pad, hi + pad], [lo - pad, hi + pad], "k--", lw=1, label="1:1")
    ax.scatter(x, y, s=70, c=np.arange(len(common_case_ids)), cmap="viridis")
    for xi, yi, case_id in zip(x, y, common_case_ids):
        ax.text(xi, yi, case_id.split("_")[0].replace("wf", ""), fontsize=8, ha="left", va="bottom")
    ax.set_title(f"{freq} Hz")
    ax.set_xlabel("deep cavity R")
    ax.grid(True, alpha=0.3)
axes[0].set_ylabel("100 m supported cavity R")
fig.suptitle("Reflection-coefficient change after adding support beneath the cavity")
fig.tight_layout()

In [ ]:
freqs = [10, 5, 1]
old_matrix = np.vstack([vec(old, freq, "reflection_coefficient") for freq in freqs]).T
new_matrix = np.vstack([vec(new, freq, "reflection_coefficient") for freq in freqs]).T
delta_matrix = new_matrix - old_matrix

fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=True)
for ax, data, title in zip(axes, [old_matrix, new_matrix, delta_matrix], ["Deep cavity", "Supported cavity", "Difference (new - old)"]):
    im = ax.imshow(data, aspect="auto", cmap="coolwarm")
    ax.set_xticks(range(len(freqs)), [f"{freq} Hz" for freq in freqs])
    ax.set_yticks(range(len(common_case_ids)), [labels_for_case(case_id) for case_id in common_case_ids])
    ax.set_title(title)
    plt.colorbar(im, ax=ax, shrink=0.8)
fig.tight_layout()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
for case_id in common_case_ids:
    old_curve = [float(old[freq][case_id]["reflection_coefficient"]) for freq in freqs]
    new_curve = [float(new[freq][case_id]["reflection_coefficient"]) for freq in freqs]
    ax.plot(freqs, old_curve, "o--", alpha=0.7, label=f"{case_id} old")
    ax.plot(freqs, new_curve, "o-", lw=2, label=f"{case_id} new")
ax.set_xscale("log")
ax.set_xticks(freqs, [str(freq) for freq in freqs])
ax.set_xlabel("source frequency (Hz)")
ax.set_ylabel("estimated reflection coefficient")
ax.set_title("Frequency dependence for the matched cases")
ax.grid(True, alpha=0.3)
ax.legend(ncol=2, fontsize=8)
fig.tight_layout()

In [ ]:
def load_gather(results_root, freq, case_id):
    path = ROOT / results_root / f"results_{freq}hz" if results_root == "" else ROOT / f"{results_root}_{freq}hz"
    data = np.load(path / case_id / "surface_gather.npz", allow_pickle=True)
    return {key: data[key] for key in data.files}

def trace_at_x(gather, x_target):
    idx = int(np.argmin(np.abs(gather["x"] - x_target)))
    return idx, gather["x"][idx], gather["bxx"][idx, :]

selected_cases = common_case_ids
selected_x = -1000.0
fig, axes = plt.subplots(len(freqs), len(selected_cases), figsize=(4 * len(selected_cases), 3.2 * len(freqs)), sharex=True, sharey=True)
for i, freq in enumerate(freqs):
    for j, case_id in enumerate(selected_cases):
        ax = axes[i, j]
        old_g = load_gather("", freq, case_id)
        new_g = load_gather("results_supportedcavity", freq, case_id)
        idx_old, x_old, old_trace = trace_at_x(old_g, selected_x)
        idx_new, x_new, new_trace = trace_at_x(new_g, selected_x)
        scale = max(np.max(np.abs(old_trace)), np.max(np.abs(new_trace)), 1e-12)
        ax.plot(old_g["time"], old_trace / scale, label=f"deep cavity @ {x_old:.1f} m", alpha=0.8)
        ax.plot(new_g["time"], new_trace / scale, label=f"supported cavity @ {x_new:.1f} m", alpha=0.8)
        if i == 0:
            ax.set_title(case_id)
        if j == 0:
            ax.set_ylabel(f"{freq} Hz\nnormalized BXX")
        ax.grid(True, alpha=0.25)
        if i == len(freqs) - 1:
            ax.set_xlabel("time (s)")
        if i == 0 and j == 0:
            ax.legend(fontsize=8)
fig.suptitle("Matched left-side traces near x = -1000 m")
fig.tight_layout()